# Lighthouse Notebook 2: TF-IDF and logistic regression

Each of you is building a different model this week on the same data. At Build Night 3
we put the four score tables next to each other and pick what the project uses.

The data is different from last week. The `other` bucket is gone, `none` is in, and
there is now text pulled off real screenshots alongside the Mathur sentences.

The setup cells load the data. How you get from there to a working model is up to you.
You have AI tools available. Use them to write code, not to decide what to do.

Turn on the GPU before you start: Runtime, Change runtime type, T4 GPU.

## Setup

Run these three. Upload `contextdp_ocr.csv` from the shared folder first, using the file
browser on the left.

In [72]:
URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

import pandas as pd, numpy as np

mathur = pd.read_csv(URL)
mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)
mathur = mathur.dropna(subset=["label", "Pattern String"]).copy()
mathur["text"] = mathur["Pattern String"].str.strip()
mathur = mathur[mathur["text"] != ""].drop_duplicates("text")
mathur["site"] = (mathur["Website Page"].astype(str)
                  .str.extract(r"https?://([^/]+)")[0]
                  .str.replace(r"^www\.", "", regex=True))
mathur = mathur.dropna(subset=["site"])[["text", "label", "site"]].reset_index(drop=True)

print(len(mathur), "sentences from", mathur["site"].nunique(), "websites")
print(mathur["label"].value_counts())

1081 sentences from 792 websites
label
scarcity              418
social_proof          312
urgency               210
confirmshaming        129
hidden_information     12
Name: count, dtype: int64


In [73]:
# 440 screenshots from ContextDP, labelled with the mapping we built at Build Night 2,
# with the text already pulled off them by OCR.

shots = pd.read_csv("contextdp_ocr.csv")
print(len(shots), "screenshots")
print(shots["label"].value_counts())
shots.head(3)

440 screenshots
label
none                  237
preselection           98
urgency                43
hidden_information     34
scarcity               22
social_proof            6
Name: count, dtype: int64


,image,platform,their_labels,label,text
0,droom.in.png,web,ACTIVITY MESSAGE,social_proof,dr ©O iis Host Tasted 8 © location Vv (a) we #...
1,kiiroo.com.png,web,ACTIVITY MESSAGE,social_proof,Titan & Cliona Couple Set $229 or 4 interest-f...
2,nutrabay.com.png,web,ACTIVITY MESSAGE,social_proof,NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss W...


In [74]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),
    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),
    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),
    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),
    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),
    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),
    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),
    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),
    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),
    ("Everyone in dallas is buying the new ford bronco, now is your time", "social_proof"),
    ("The buy one get one free deal will end in 3 hours, shop now", "urgency"),
    ("the new harry potter movie is dropping on 22 september this year", "none"),
    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),
    ("We use cookies. Accept all or manage your preferences.", "none"),
    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(len(held_out), "held out sentences,", (held_out.label == "none").sum(), "of them none")

47 held out sentences, 23 of them none


## 1. Split the data

Use these exact settings. They are the one thing that has to match across the four of
us, otherwise our numbers are not comparable.

Sentences: grouped split on the `site` column, 20% test, `random_state=42`. Every
sentence from a website goes entirely into train or entirely into test.

Screenshots: 60/40 split, stratified on label, `random_state=42`.

Your training set is the training half of the sentences plus the training half of the
screenshots, stacked into one table of `text` and `label`.

In [75]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(mathur, groups=mathur["site"]))
mathur_train = mathur.iloc[train_idx]
mathur_test = mathur.iloc[test_idx]

shots_train, shots_test = train_test_split(
    shots, test_size=0.4, stratify=shots["label"], random_state=42
)

train_df = pd.concat([
    mathur_train[["text", "label"]],
    shots_train[["text", "label"]]
], ignore_index=True)

print(len(mathur_train), "mathur train,", len(mathur_test), "mathur test")
print(len(shots_train), "shots train,", len(shots_test), "shots test")
print(len(train_df), "total training rows")

863 mathur train, 218 mathur test
264 shots train, 176 shots test
1127 total training rows


## 2. Build TF-IDF plus logistic regression

Same shape as Notebook 1: TF-IDF into logistic regression. Two settings you want on the
classifier, `max_iter=1000` and `class_weight="balanced"`, since our labels are lopsided.

When you predict, pass the text, never the labels.

If you have time once everything else runs, try letting TF-IDF see word pairs as well as
single words, and report whether it helped.

In [76]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
pipe.fit(train_df["text"], train_df["label"])

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

## 3. Look at the OCR text

Print the label and the first 200 characters of `text` for five screenshots, and read
them. You need this for question 3.

In [77]:
for _, row in shots.sample(5, random_state=1).iterrows():
    print(row["label"], "-", row["text"][:200])
    print()

none - CHINA | TV" a Shuttle Love Millennium Info Videos Reviews Comments Volunteers Can you get your life back in 80 years? Sun Qi Long (Wei Da Xun) is a rich party boy living in 2016. But his life is turne

hidden_information - 9 %® 6 6 09:54 o @ - XY Countdown 00:10:00 Know Better About Yourself NOW! + = ee) 3-Day Free Trial aT

none - Online Now shelbie, 179 mi ae ‘ ate O

none - 2H O44 ® v7 \ 4:59 Create New Project

preselection - 20:07 NER oo ee i: 592541 > Risingstar29023900 jc @ ‘Alarms & Clock - a 0 ®@ 6 6 00:02:13 > @~ ‘Countdown 00:10:00 zt ii t



## 4. Score it three ways

Print a `classification_report` for each:

1. the sentence test set
2. the screenshot test set
3. `held_out`

In [78]:
from sklearn.metrics import classification_report

pred_sentence = pipe.predict(mathur_test["text"])
print("Sentence test set:")
print(classification_report(mathur_test["label"], pred_sentence))

pred_screenshot = pipe.predict(shots_test["text"])
print("Screenshot test set:")
print(classification_report(shots_test["label"], pred_screenshot))

pred_heldout = pipe.predict(held_out["text"])
print("Held out:")
print(classification_report(held_out["label"], pred_heldout))

Sentence test set:
                    precision    recall  f1-score   support

    confirmshaming       1.00      0.96      0.98        28
hidden_information       0.00      0.00      0.00         1
              none       0.00      0.00      0.00         0
          scarcity       1.00      0.92      0.96        84
      social_proof       0.99      0.97      0.98        68
           urgency       0.88      0.95      0.91        37

          accuracy                           0.94       218
         macro avg       0.64      0.63      0.64       218
      weighted avg       0.97      0.94      0.95       218

Screenshot test set:
                    precision    recall  f1-score   support

hidden_information       0.62      0.57      0.59        14
              none       0.78      0.87      0.82        95
      preselection       0.71      0.74      0.72        39
          scarcity       1.00      0.11      0.20         9
      social_proof       0.00      0.00      0.00       

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

## 5. Find what it got wrong

Build a small dataframe of the `held_out` sentences your model missed, with the true
label and the prediction, and print it. Bring this to Build Night 3.

In [79]:
wrong_mask = pred_heldout != held_out["label"].values
wrong_df = pd.DataFrame({
    "text": held_out["text"][wrong_mask],
    "true_label": held_out["label"][wrong_mask],
    "predicted": pred_heldout[wrong_mask]
})
print(wrong_df)

                                                 text      true_label  \
1                    Buy it before the timer runs out         urgency   
2             There are hardly any of these remaining        scarcity   
3              You would be missing out if you say no  confirmshaming   
6   The performance of the product is better now, ...        scarcity   
7   Are you sure you want to stop protecting your ...  confirmshaming   
9                                           1000 left            none   
11              between 0 and 1 million left in stock            none   
24    25 people have this in their cart at the moment        scarcity   
28         70% of our users are making more than 700K    social_proof   
29  Everyone in dallas is buying the new ford bron...    social_proof   
32                   Free shipping on orders over $50            none   
33                             Currently out of stock            none   
35                   Summer sale: 20% off all sanda

## 6. Save three files

Use these names and columns so they merge:

`results_YOURNAME.csv` with columns `model, eval_set, accuracy, macro_f1`, one row per
eval set. Model name for this notebook is `tfidf_logreg`.

`wrong_YOURNAME.csv`, the dataframe from step 5.

`preds_YOURNAME.csv`, your predictions on the screenshot test set, with columns
`image, label, predicted`.

Download all three from the file browser and put them in the shared folder.

In [80]:
from sklearn.metrics import accuracy_score, f1_score

results = pd.DataFrame([
    {"model": "tfidf_logreg", "eval_set": "sentence",
     "accuracy": accuracy_score(mathur_test["label"], pred_sentence),
     "macro_f1": f1_score(mathur_test["label"], pred_sentence, average="macro")},
    {"model": "tfidf_logreg", "eval_set": "screenshot",
     "accuracy": accuracy_score(shots_test["label"], pred_screenshot),
     "macro_f1": f1_score(shots_test["label"], pred_screenshot, average="macro")},
    {"model": "tfidf_logreg", "eval_set": "held_out",
     "accuracy": accuracy_score(held_out["label"], pred_heldout),
     "macro_f1": f1_score(held_out["label"], pred_heldout, average="macro")},
])
results.to_csv("results_Khushi.csv", index=False)

In [81]:
wrong_df.to_csv("wrong_Khushi.csv", index=False)

In [82]:
print(shots.columns.tolist())

['image', 'platform', 'their_labels', 'label', 'text']


In [83]:
preds_df = pd.DataFrame({
    "image": shots_test["image"],
    "label": shots_test["label"],
    "predicted": pred_screenshot
})
preds_df.to_csv("preds_Khushi.csv", index=False)

In [84]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 7. Questions

1. Your three accuracy numbers.

Sentence: 94% | Screenshot: 75% | Held out: 57%


2. Why is the screenshot score different from the sentence score?

Screenshot text is noisy OCR output (garbled, mixed with unrelated UI elements), while sentence text is clean and isolated — so the model performs worse on messier, less familiar input.


3. How is OCR text different from a Mathur sentence? One line.

OCR text is jumbled and unstructured, often mixing multiple unrelated UI elements and misread characters into one blob, unlike Mathur's clean, isolated dark pattern phrases.

4. Two held out sentences you got wrong, with what the model guessed. Was one of them a
   `none`?

Pull two rows straight from your wrong_df output (the one from Step 5) — pick any two, note the true label vs. predicted, and flag if either true label was none.
   
5. Which category does your model handle worst on the screenshots, and what would you need to see to catch it?

Scarcity — on the screenshot set it had 100% precision but only 11% recall (f1 = 0.20), meaning it almost never predicted scarcity even when it should have. To catch it, you'd need more real, varied scarcity examples in screenshot form (not just Mathur-style ones) so the model learns what scarcity actually looks like in messy OCR text.

6. One thing that confused you.

I was confused about why class_weight='balanced' was necessary given TfidfVectorizer already handles feature weighting

7. Guess how your model does against the other three. We check next week.

I think logistic regression will land in the middle, likely beating simpler models like MNB but behind anything using more contextual features





## Submitting

1. Runtime, then Restart session and run all. Wait for it to finish.
2. Scroll through and check every cell has its output showing. If a cell is empty, your
   results did not save.
3. File, Download, Download .ipynb.
4. Post the notebook and your three CSVs in the shared folder before Build Night 3.

If something crashes halfway, fix it and run all again. A notebook with no outputs is
not gradeable and your model drops out of the comparison.

In [85]:
screens_split = pd.read_csv("screens_split.csv")
shots_merged = shots.merge(screens_split, on="image")
shots_train = shots_merged[shots_merged["split"] == "train"]
shots_test = shots_merged[shots_merged["split"] == "test"]

In [86]:
train_df_v2 = pd.concat([
    mathur_train[["text", "label"]],
    shots_train[["text", "label"]]
], ignore_index=True)

pipe.fit(train_df_v2["text"], train_df_v2["label"])

pred_sentence = pipe.predict(mathur_test["text"])
pred_screenshot = pipe.predict(shots_test["text"])
pred_heldout = pipe.predict(held_out["text"])

In [87]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
FOUR = ["confirmshaming", "scarcity", "social_proof", "urgency"]

def score_all(model_name, y_s, p_s, y_sh, p_sh, y_h, p_h):
    y_s, p_s, y_sh, p_sh, y_h, p_h = map(np.asarray, (y_s, p_s, y_sh, p_sh, y_h, p_h))
    print(classification_report(y_sh, p_sh, zero_division=0))
    rows = {
        "sent_f1_four":        f1_score(y_s, p_s, labels=FOUR, average="macro", zero_division=0),
        "shot_accuracy":       accuracy_score(y_sh, p_sh),
        "shot_always_none":    (y_sh == "none").mean(),
        "shot_dark_f1":        f1_score(y_sh != "none", p_sh != "none"),
        "heldout_none_passed": int(((y_h == "none") & (p_h == "none")).sum()),
        "heldout_dark_right":  int(((y_h != "none") & (p_h == y_h)).sum()),
    }
    return pd.DataFrame([{"model": model_name, "metric": k, "value": v} for k, v in rows.items()])

results_v2 = score_all(
    "tfidf_baseline",
    mathur_test["label"], pred_sentence,
    shots_test["label"], pred_screenshot,
    held_out["label"], pred_heldout
)
results_v2.to_csv("results_Khushi_v2.csv", index=False)

preds_v2 = pd.DataFrame({
    "image": shots_test["image"],
    "label": shots_test["label"],
    "predicted": pred_screenshot
})
preds_v2.to_csv("preds_Khushi_v2.csv", index=False)

                    precision    recall  f1-score   support

    confirmshaming       0.00      0.00      0.00         0
hidden_information       0.50      0.21      0.30        19
              none       0.75      0.89      0.81        98
      preselection       0.59      0.69      0.64        32
          scarcity       0.00      0.00      0.00         5
      social_proof       0.00      0.00      0.00         3
           urgency       0.67      0.47      0.55        17

          accuracy                           0.70       174
         macro avg       0.36      0.32      0.33       174
      weighted avg       0.65      0.70      0.66       174



In [88]:
import re

def classify_filename(fname):
    if "--" in fname:
        return "app_screen"
    if re.match(r"^\(\d+\)", fname):
        return "numbered_site"
    if fname.endswith(".png"):
        return "site_png"
    if re.match(r"^\d+_[a-z0-9]+\.", fname, re.IGNORECASE):
        return "number_code"
    if re.match(r"^\d+\.", fname):
        return "plain_number"
    return "other"

shots["file_type"] = shots["image"].apply(classify_filename)

In [89]:
print(pd.crosstab(shots["file_type"], shots["label"]))

label          hidden_information  none  preselection  scarcity  social_proof  \
file_type                                                                       
app_screen                     19     0            55         0             0   
number_code                     7     1            27         0             0   
numbered_site                   4     0             8         9             1   
other                           0     0             0         2             0   
plain_number                    4   157             8         0             0   
site_png                        0    79             0        11             5   

label          urgency  
file_type               
app_screen           0  
number_code          0  
numbered_site       27  
other                0  
plain_number         0  
site_png            16  


In [90]:
# Most common label per file type, using the new split's train half
most_common = shots_train.groupby(
    shots_train["image"].apply(classify_filename)
)["label"].agg(lambda x: x.value_counts().idxmax())

shots_test_types = shots_test["image"].apply(classify_filename)
pred_filename_new = shots_test_types.map(most_common)
acc_filename_new = (pred_filename_new == shots_test["label"]).mean()
print("New split, filename rule accuracy:", acc_filename_new)

New split, filename rule accuracy: 0.7931034482758621


In [91]:
shots_old_train, shots_old_test = train_test_split(
    shots, test_size=0.4, stratify=shots["label"], random_state=42
)

most_common_old = shots_old_train.groupby(
    shots_old_train["image"].apply(classify_filename)
)["label"].agg(lambda x: x.value_counts().idxmax())

shots_old_test_types = shots_old_test["image"].apply(classify_filename)
pred_filename_old = shots_old_test_types.map(most_common_old)
acc_filename_old = (pred_filename_old == shots_old_test["label"]).mean()
print("Old split, filename rule accuracy:", acc_filename_old)

Old split, filename rule accuracy: 0.7897727272727273


In [92]:
filename_rule = pd.DataFrame([
    {"split": "old", "method": "filename_rule", "accuracy": acc_filename_old},
    {"split": "old", "method": "tfidf", "accuracy": 0.75},
    {"split": "new", "method": "filename_rule", "accuracy": acc_filename_new},
    {"split": "new", "method": "tfidf", "accuracy": results_v2.loc[results_v2["metric"] == "shot_accuracy", "value"].values[0]},
])
filename_rule.to_csv("filename_rule.csv", index=False)
print(filename_rule)

  split         method  accuracy
0   old  filename_rule  0.789773
1   old          tfidf  0.750000
2   new  filename_rule  0.793103
3   new          tfidf  0.695402


In [93]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [94]:
import os
base = "/content/drive/MyDrive/Lighthouse F26"
for person in ["Prasanna", "Muhammad", "Khushi", "Arjun"]:
    print(person, "-", len(os.listdir(os.path.join(base, person))), "files")

Prasanna - 43 files
Muhammad - 43 files
Khushi - 45 files
Arjun - 3 files


In [95]:
names = ["Arjun", "Khushi", "Muhammad", "Prasanna"]
dfs = []
for n in names:
    d = pd.read_csv(f"Data collection - {n}.csv")
    d["collector"] = n
    dfs.append(d)

cohort = pd.concat(dfs, ignore_index=True)
print(cohort.shape)
print(cohort.columns.tolist())
cohort.head()

(156, 10)
['File Name', 'Category(s)', 'Justification (for each category selected)', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'collector']


,File Name,Category(s),Justification (for each category selected),Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,collector
0,Arjun_01,Preselection,Pop up to add items to your order that You wer...,NaN,NaN,NaN,NaN,NaN,NaN,Arjun
1,Arjun_02,Hidden information,Promises a price and discount throughout the w...,NaN,Category,Count,Minimum,Cap,Status,Arjun
2,Arjun_02,Preselection,Promises a price and discount throughout the w...,NaN,None (hard negatives),10,12,17,Need 2 more,Arjun
3,Arjun_03,Urgency,Coundown timer on a sale that was there yester...,NaN,Confirmshaming,0,10,15,Need 10 more,Arjun
4,Arjun_04,Urgency,"""dont miss out"" and ""offer ends in 3 days"" on ...",NaN,Preselection,2,6,10,Need 4 more,Arjun


In [96]:
cohort = cohort[["File Name", "Category(s)", "Justification (for each category selected)", "collector"]]
cohort.columns = ["file", "category", "justification", "collector"]
cohort = cohort.dropna(subset=["file"])
print(cohort.shape)
cohort.head(10)

(156, 4)


,file,category,justification,collector
0,Arjun_01,Preselection,Pop up to add items to your order that You wer...,Arjun
1,Arjun_02,Hidden information,Promises a price and discount throughout the w...,Arjun
2,Arjun_02,Preselection,Promises a price and discount throughout the w...,Arjun
3,Arjun_03,Urgency,Coundown timer on a sale that was there yester...,Arjun
4,Arjun_04,Urgency,"""dont miss out"" and ""offer ends in 3 days"" on ...",Arjun
5,Arjun_05,Urgency,5 min countdown timer on the deal they offer y...,Arjun
6,Arjun_06,Urgency,"Its always the ""last chance"" to get a gifcard ...",Arjun
7,Arjun_07,Social proof,"""overall pick"" and ""10k bought last month""",Arjun
8,Arjun_08,Social proof,"""Hey this guy you know has premium you should ...",Arjun
9,Arjun_09,Urgency,Big countdown timer to create a sense of urgency,Arjun


In [97]:
SPELLING = {
    "Urgency": "urgency",
    "Scarcity": "scarcity",
    "Social proof": "social_proof",
    "Confirmshaming": "confirmshaming",
    "Preselection": "preselection",
    "Hidden information": "hidden_information",
    "None": "none",
}

cohort["category_clean"] = cohort["category"].map(SPELLING)

# anything that didn't map is a problem — check before moving on
bad = cohort[cohort["category_clean"].isna()]
print("Unmapped categories:", bad["category"].unique())

Unmapped categories: [nan]


In [98]:
no_label_rows = cohort[cohort["category"].isna()]
print(len(no_label_rows), "rows with no category")
no_label_rows[["file", "collector"]]

46 rows with no category


,file,collector
11,Arjun_11_Negative,Arjun
12,Arjun_12_Negative,Arjun
13,Arjun_13_Negative,Arjun
14,Arjun_14_Negative,Arjun
15,Arjun_15_Negative,Arjun
16,Arjun_16_Negative,Arjun
18,Arjun_18_Negative,Arjun
19,Arjun_19_Negative,Arjun
20,Arjun_20_Negative,Arjun
21,Arjun_21_Negative,Arjun


In [99]:
no_label_rows[["file", "collector"]].to_csv("label_issues_no_label.csv", index=False)
with open("label_issues.txt", "w") as f:
    f.write("=== Rows with no label ===\n")
    for _, row in no_label_rows.iterrows():
        f.write(f"{row['collector']}: {row['file']}\n")

In [100]:
cohort_labeled = cohort.dropna(subset=["category_clean"]).copy()

In [101]:
PRIORITY = ["confirmshaming", "hidden_information", "preselection", "scarcity", "urgency", "social_proof"]

def pick_main(labels):
    for p in PRIORITY:
        if p in labels:
            return p
    return labels[0]

grouped = cohort_labeled.groupby("file").agg(
    all_labels_list=("category_clean", list),
    justification=("justification", lambda x: "|".join(x.dropna().astype(str))),
    collector=("collector", "first"),
).reset_index()

grouped["all_labels"] = grouped["all_labels_list"].apply(lambda x: ";".join(x))
grouped["label"] = grouped["all_labels_list"].apply(pick_main)
grouped = grouped.drop(columns=["all_labels_list"])

print(grouped.shape)
grouped.head(10)

(107, 5)


,file,justification,collector,all_labels,label
0,Arjun_01,Pop up to add items to your order that You wer...,Arjun,preselection,preselection
1,Arjun_02,Promises a price and discount throughout the w...,Arjun,hidden_information;preselection,hidden_information
2,Arjun_03,Coundown timer on a sale that was there yester...,Arjun,urgency,urgency
3,Arjun_04,"""dont miss out"" and ""offer ends in 3 days"" on ...",Arjun,urgency,urgency
4,Arjun_05,5 min countdown timer on the deal they offer y...,Arjun,urgency,urgency
5,Arjun_06,"Its always the ""last chance"" to get a gifcard ...",Arjun,urgency,urgency
6,Arjun_07,"""overall pick"" and ""10k bought last month""",Arjun,social_proof,social_proof
7,Arjun_08,"""Hey this guy you know has premium you should ...",Arjun,social_proof,social_proof
8,Arjun_09,Big countdown timer to create a sense of urgency,Arjun,urgency,urgency
9,Arjun_10,"""only 6 left in stock""",Arjun,scarcity,scarcity


In [102]:
image_files = []
for person in names:
    folder = os.path.join(base, person)
    for fname in os.listdir(folder):
        image_files.append({"image_file": fname, "folder": person})

images_df = pd.DataFrame(image_files)
print(images_df.shape)
images_df.head()

(134, 2)


,image_file,folder
0,Assignment 1,Arjun
1,Week1_Screenshots,Arjun
2,Week2_Data_Collection,Arjun
3,Khushi_01.png,Khushi
4,Khushi_02.png,Khushi


In [103]:
image_files = []
for person in names:
    folder = os.path.join(base, person)
    for root, dirs, files in os.walk(folder):
        for fname in files:
            image_files.append({"image_file": fname, "folder": person})

images_df = pd.DataFrame(image_files)
print(images_df.shape)
print(images_df["folder"].value_counts())
images_df.head()

(155, 2)
folder
Khushi      45
Muhammad    43
Prasanna    43
Arjun       24
Name: count, dtype: int64


,image_file,folder
0,LighthouseAssignment1.ipynb,Arjun
1,Arjun_01.png,Arjun
2,Arjun_02.png,Arjun
3,Arjun_03.png,Arjun
4,Arjun_04.png,Arjun


In [104]:
images_df = images_df[images_df["image_file"].str.lower().str.endswith((".png", ".jpg", ".jpeg"))].copy()
print(images_df.shape)
print(images_df["folder"].value_counts())

(153, 2)
folder
Khushi      45
Muhammad    43
Prasanna    43
Arjun       22
Name: count, dtype: int64


In [105]:
images_df["file_noext"] = images_df["image_file"].str.replace(r"\.(png|jpg|jpeg)$", "", regex=True, case=False)

matched = grouped.merge(images_df, left_on="file", right_on="file_noext", how="outer", indicator=True)

sheet_no_image = matched[matched["_merge"] == "left_only"]
image_no_sheet = matched[matched["_merge"] == "right_only"]
both_matched = matched[matched["_merge"] == "both"]

print("Matched:", len(both_matched))
print("Sheet rows with no image:", len(sheet_no_image))
print("Images with no sheet row:", len(image_no_sheet))

Matched: 73
Sheet rows with no image: 34
Images with no sheet row: 80


In [106]:
print("--- Sheet rows with no image (sample) ---")
print(sheet_no_image[["file", "collector"]].head(20))

print("--- Images with no sheet row (sample) ---")
print(image_no_sheet[["image_file", "folder"]].head(20))

--- Sheet rows with no image (sample) ---
              file collector
22   Khushi.09.png    Khushi
24   Khushi_01.png    Khushi
26   Khushi_02.png    Khushi
28   Khushi_03.png    Khushi
30   Khushi_04.png    Khushi
32   Khushi_05.png    Khushi
34   Khushi_06.png    Khushi
36   Khushi_07.png    Khushi
38   Khushi_08.png    Khushi
41   Khushi_10.png    Khushi
43   Khushi_11.png    Khushi
44  Khushi_11.png     Khushi
46   Khushi_12.png    Khushi
48   Khushi_13.png    Khushi
50   Khushi_14.png    Khushi
52   Khushi_15.png    Khushi
54   Khushi_16.png    Khushi
56   Khushi_17.png    Khushi
58   Khushi_18.png    Khushi
60   Khushi_19.png    Khushi
--- Images with no sheet row (sample) ---
                   image_file  folder
10      Arjun_11_Negative.png   Arjun
11      Arjun_12_Negative.png   Arjun
12      Arjun_13_Negative.png   Arjun
13      Arjun_14_Negative.png   Arjun
14      Arjun_15_Negative.png   Arjun
15  Arjun_15_Negative (1).png   Arjun
16      Arjun_16_Negative.png   Arjun
18 

In [107]:
grouped["file_noext"] = grouped["file"].str.replace(r"\.(png|jpg|jpeg)$", "", regex=True, case=False)

matched = grouped.merge(images_df, left_on="file_noext", right_on="file_noext", how="outer", indicator=True)

sheet_no_image = matched[matched["_merge"] == "left_only"]
image_no_sheet = matched[matched["_merge"] == "right_only"]
both_matched = matched[matched["_merge"] == "both"]

print("Matched:", len(both_matched))
print("Sheet rows with no image:", len(sheet_no_image))
print("Images with no sheet row:", len(image_no_sheet))

Matched: 105
Sheet rows with no image: 2
Images with no sheet row: 48


In [108]:
print("--- Sheet rows with no image ---")
print(sheet_no_image[["file", "collector"]])

print("--- Images with no sheet row ---")
print(image_no_sheet[["image_file", "folder"]].to_string())

--- Sheet rows with no image ---
              file collector
22   Khushi.09.png    Khushi
34  Khushi_11.png     Khushi
--- Images with no sheet row ---
                    image_file    folder
10       Arjun_11_Negative.png     Arjun
11       Arjun_12_Negative.png     Arjun
12       Arjun_13_Negative.png     Arjun
13       Arjun_14_Negative.png     Arjun
14       Arjun_15_Negative.png     Arjun
15   Arjun_15_Negative (1).png     Arjun
16       Arjun_16_Negative.png     Arjun
18       Arjun_18_Negative.png     Arjun
19       Arjun_19_Negative.png     Arjun
20       Arjun_20_Negative.png     Arjun
21       Arjun_21_Negative.png     Arjun
31               Khushi_09.png    Khushi
43               Khushi_20.png    Khushi
58               Khushi_35.png    Khushi
59               Khushi_36.png    Khushi
60               Khushi_37.png    Khushi
61               Khushi_38.png    Khushi
62               Khushi_39.png    Khushi
63               Khushi_40.png    Khushi
64               Khushi_41.

In [109]:
with open("label_issues.txt", "a") as f:
    f.write("\n=== Sheet rows with no image ===\n")
    for _, row in sheet_no_image.iterrows():
        f.write(f"{row['collector']}: {row['file']}\n")

    f.write("\n=== Images with no sheet row ===\n")
    for _, row in image_no_sheet.iterrows():
        f.write(f"{row['folder']}: {row['image_file']}\n")

print("label_issues.txt updated")

label_issues.txt updated


In [110]:
print(both_matched["collector"].value_counts())

collector
Khushi      32
Prasanna    31
Muhammad    31
Arjun       11
Name: count, dtype: int64


In [111]:
with open("make_contextdp_ocr.py") as f:
    print(f.read())

# make_contextdp_ocr.py
# Downloads ContextDP, applies the Lighthouse mapping, OCRs every screenshot,
# writes contextdp_ocr.csv  (image, platform, their_labels, label, text)
#
# Needs: pip install pytesseract pandas pillow
#        apt-get install -y tesseract-ocr
#
# First, download and unpack the dataset (~130 MB):
#   curl -sL -o contextdp.tar.gz \
#     "https://github.com/SageSELab/AidUI/releases/download/v-ICSE'23-data/AidUI-Evaluation-Dataset.tar.gz"
#   mkdir -p contextdp && tar xzf contextdp.tar.gz -C contextdp 2>/dev/null

import json
import pandas as pd
import pytesseract
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

ROOT = "contextdp/evaluation_dataset"

# ContextDP's labels -> ours. None means drop the screenshot entirely.
MAP = {
    "ACTIVITY MESSAGE":      "social_proof",
    "HIGH DEMAND MESSAGE":   "scarcity",
    "LOW STOCK MESSAGE":     "scarcity",
    "LIMITED TIME MESSAGE":  "urgency",
    "COUNTDOWN TIMER":       "urgency",
    "ATTENT

In [112]:
!pip install pytesseract pillow -q
!apt-get install -y tesseract-ocr -q

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [113]:
import pytesseract
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

def ocr(path):
    try:
        return " ".join(pytesseract.image_to_string(Image.open(path)).split())
    except Exception:
        return ""

both_matched = both_matched.copy()
both_matched["path"] = both_matched.apply(
    lambda r: os.path.join(base, r["folder"], r["image_file"]), axis=1
)

with ThreadPoolExecutor(12) as ex:
    both_matched["text"] = list(ex.map(ocr, both_matched["path"]))

empty_count = (both_matched["text"].str.len() == 0).sum()
print(empty_count, "rows came back with empty text")
print(both_matched.shape)

12 rows came back with empty text
(105, 11)


In [114]:
def get_dims(path):
    try:
        with Image.open(path) as img:
            return img.size  # (width, height)
    except Exception:
        return (None, None)

dims = both_matched["path"].apply(get_dims)
both_matched["width"] = dims.apply(lambda x: x[0])
both_matched["height"] = dims.apply(lambda x: x[1])

def classify_platform(row):
    if row["width"] is None or row["height"] is None:
        return "web"  # fallback, will be flagged below
    if row["height"] > 1.5 * row["width"]:
        return "mobile"
    return "web"

both_matched["platform"] = both_matched.apply(classify_platform, axis=1)
print(both_matched["platform"].value_counts())
print(both_matched[["image_file", "width", "height", "platform"]].head(10))


platform
web       101
mobile      4
Name: count, dtype: int64
     image_file  width  height platform
0  Arjun_01.png    NaN     NaN      web
1  Arjun_02.png    NaN     NaN      web
2  Arjun_03.png    NaN     NaN      web
3  Arjun_04.png    NaN     NaN      web
4  Arjun_05.png    NaN     NaN      web
5  Arjun_06.png    NaN     NaN      web
6  Arjun_07.png    NaN     NaN      web
7  Arjun_08.png    NaN     NaN      web
8  Arjun_09.png    NaN     NaN      web
9  Arjun_10.png    NaN     NaN      web


In [115]:
test_path = both_matched["path"].iloc[0]
print(test_path)
print(os.path.exists(test_path))

/content/drive/MyDrive/Lighthouse F26/Arjun/Arjun_01.png
False


In [116]:
image_files = []
for person in names:
    folder = os.path.join(base, person)
    for root, dirs, files in os.walk(folder):
        for fname in files:
            if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                image_files.append({
                    "image_file": fname,
                    "folder": person,
                    "full_path": os.path.join(root, fname)
                })

images_df = pd.DataFrame(image_files)
images_df["file_noext"] = images_df["image_file"].str.replace(r"\.(png|jpg|jpeg)$", "", regex=True, case=False)
print(images_df.shape)

(153, 4)


In [117]:
matched = grouped.merge(images_df, left_on="file_noext", right_on="file_noext", how="outer", indicator=True)
both_matched = matched[matched["_merge"] == "both"].copy()
print(len(both_matched))

105


In [118]:
with ThreadPoolExecutor(12) as ex:
    both_matched["text"] = list(ex.map(ocr, both_matched["full_path"]))

empty_count = (both_matched["text"].str.len() == 0).sum()
print(empty_count, "rows came back with empty text")

dims = both_matched["full_path"].apply(get_dims)
both_matched["width"] = dims.apply(lambda x: x[0])
both_matched["height"] = dims.apply(lambda x: x[1])
both_matched["platform"] = both_matched.apply(classify_platform, axis=1)

print(both_matched["platform"].value_counts())
print(both_matched[["image_file", "width", "height", "platform"]].head(10))

1 rows came back with empty text
platform
web       101
mobile      4
Name: count, dtype: int64
     image_file  width  height platform
0  Arjun_01.png   1917     882      web
1  Arjun_02.png   1901     875      web
2  Arjun_03.png   1901     876      web
3  Arjun_04.png   1917     881      web
4  Arjun_05.png   1917     862      web
5  Arjun_06.png   1917     962      web
6  Arjun_07.png   1912     877      web
7  Arjun_08.png   1917     878      web
8  Arjun_09.png   1917     877      web
9  Arjun_10.png   1917     880      web


In [119]:
empty_rows = both_matched[both_matched["text"].str.len() == 0]
print(empty_rows[["image_file", "folder", "width", "height"]])

   image_file    folder  width  height
95   LH33.png  Prasanna    987     912


In [120]:
cohort_ocr = both_matched[["image_file", "collector", "platform", "label", "all_labels", "justification", "text"]].copy()
cohort_ocr.columns = ["image", "collector", "platform", "label", "all_labels", "justification", "text"]

cohort_ocr.to_csv("cohort_ocr.csv", index=False)
print(cohort_ocr.shape)
cohort_ocr.head()

(105, 7)


,image,collector,platform,label,all_labels,justification,text
0,Arjun_01.png,Arjun,web,preselection,preselection,Pop up to add items to your order that You wer...,x Complete your order Dr Pepper $2.70 Popular ...
1,Arjun_02.png,Arjun,web,hidden_information,hidden_information;preselection,Promises a price and discount throughout the w...,Meivery Veils i Send asa gift > ae Add a perso...
2,Arjun_03.png,Arjun,web,urgency,urgency,Coundown timer on a sale that was there yester...,{-) Free Shipping by Amazon Get FREE Shipping ...
3,Arjun_04.png,Arjun,web,urgency,urgency,"""dont miss out"" and ""offer ends in 3 days"" on ...",Sem © InstallApp LL 48 Cm) ich] (=) (Q) ~=.Q W...
4,Arjun_05.png,Arjun,web,urgency,urgency,5 min countdown timer on the deal they offer y...,"Pickup or delivery? Ca) Reorder Hi, arjunS Ss ..."


In [121]:
cohort_ocr.to_csv(os.path.join(base, "cohort_ocr.csv"), index=False)

In [122]:
spot_check = cohort_ocr.sample(10, random_state=1)
for _, row in spot_check.iterrows():
    print(row["image"], "-", row["text"][:200])
    print()

LH40.png - LIVE BEARDED FP lo oo oe oo oe SHOP Vv ABOUT v ALL NATURAL DEODORANT ‘ALL-DAY PROTECTION HELP v BLOG REVIEWS (SEARCH FRAGRANCE S e ] AMERICAN TOMBSTONE LEGEND 1880 Cedar, Vetiver, Sweet Tobacco & Sand

Khushi_27.png - DON'T GO YET! HERE'S $20 OFF YOUR $40 STYLING FEE. USE CODE: GETSTYLED20 Our stylists are eager to get to know you and style the perfect box for you. Enter your email to get styled today. EMAIL ADDRES

Khushi_34.png - Statement | Debit Card | Services Exclusive Offer For You Enjoy seamless transactions with ICICI Bank's PayLater for UPI, scan QR and sy, Net Banking and pay us back within 45 days! Now you can also c

Muhammad_32.png - Explore current top deals Check out our best deals across our product range, and get what your website needs. Free LLC | Business Starter Kit @ Start your business for free: -LLC -Domain -Website -Mar

Khushi_30.png - Order Subtotal $50.98 Standard Delivery $14.99 Tax $4.56 Total $73.52 Savings Today @ $9.00 Get a Delivery Rebate u

In [123]:
ocr_check_lines = [
    "LH40.png: missing text",
    "Khushi_27.png: missing text",
    "Khushi_34.png: missing text",
    "Muhammad_32.png: OK",
    "Khushi_30.png: OK",
    "Khushi_31.png: OK",
    "LH30.png: missing text",
    "Muhammad_13.png: OK",
    "Khushi_23.png: OK",
    "Muhammad_04.png: OK",
]
with open("ocr_check.txt", "w") as f:
    f.write("\n".join(ocr_check_lines))

print("ocr_check.txt saved")

ocr_check.txt saved
